<a href="https://www.kaggle.com/code/murtazaabdullah2010/neoai-2026-day-1-kaggleforces-code?scriptVersionId=329226615" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/neoai-2026-day-1-kaggleforces/sample_submission.csv
/kaggle/input/competitions/neoai-2026-day-1-kaggleforces/train.parquet
/kaggle/input/competitions/neoai-2026-day-1-kaggleforces/test.csv


In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
train_ds = pd.read_parquet("/kaggle/input/competitions/neoai-2026-day-1-kaggleforces/train.parquet")
test_ds = pd.read_csv("/kaggle/input/competitions/neoai-2026-day-1-kaggleforces/test.csv")

train_ds.head()

,UserId,CompetitionId,HostSegmentTitle,EnabledDate,DeadlineDate,TotalTeams,PublicLeaderboardRank,PrivateLeaderboardRank,final_public_score,final_private_score,n_submissions,Medal,RewardType,RewardQuantity,EvaluationAlgorithmIsMax
0,254908,4323,Featured,2010-04-07 07:57:43,2010-05-25 18:00:00,22,4.0,7.0,0.0,2626.0,1,NaN,USD,1000.0000,False
1,407156,4323,Featured,2010-04-07 07:57:43,2010-05-25 18:00:00,22,5.0,13.0,0.0,2830.0,1,NaN,USD,1000.0000,False
2,447896,4323,Featured,2010-04-07 07:57:43,2010-05-25 18:00:00,22,17.0,10.0,0.0,2752.0,1,NaN,USD,1000.0000,False
3,478232,4323,Featured,2010-04-07 07:57:43,2010-05-25 18:00:00,22,19.0,6.0,0.0,2612.0,1,NaN,USD,1000.0000,False
4,208298,4323,Featured,2010-04-07 07:57:43,2010-05-25 18:00:00,22,21.0,1.0,0.0,2204.0,1,NaN,USD,1000.0000,False


In [11]:
train_ds["RewardQuantity"] = train_ds["RewardQuantity"].replace("", np.nan).astype(float).fillna(0)
train_ds["log_reward"] = np.log1p(train_ds["RewardQuantity"])
train_ds["log_teams"] = np.log1p(train_ds["TotalTeams"])
train_ds["top3_rate_flag"] = (train_ds["PrivateLeaderboardRank"] <= 3).astype(int)
train_ds["gold_rate_flag"] = (train_ds["PrivateLeaderboardRank"] == 1).astype(int)
user_feat = train_ds.groupby("UserId").agg(
    n_competitions=("CompetitionId", "nunique"),
    mean_rank_pct=("PrivateLeaderboardRank", "mean"),
    median_rank_pct=("PrivateLeaderboardRank", "median"),
    best_rank_pct=("PrivateLeaderboardRank", "min"),
    last_rank_pct=("PrivateLeaderboardRank", "last"),
    avg_submissions=("n_submissions", "mean"),
    top3_rate=("top3_rate_flag", "mean"),
    gold_rate=("gold_rate_flag", "mean"),
).reset_index()
comp_feat = train_ds.groupby("CompetitionId").agg(
    comp_mean_rank=("PrivateLeaderboardRank", "mean"),
    comp_median_rank=("PrivateLeaderboardRank", "median"),
    comp_size=("UserId", "count"),
    comp_avg_submissions=("n_submissions", "mean"),
    comp_reward=("log_reward", "mean"),
    comp_team_size=("log_teams", "mean"),
).reset_index()
train = train_ds.merge(user_feat, on="UserId", how="left").merge(comp_feat, on="CompetitionId", how="left").fillna(0)
test = test_ds.merge(user_feat, on="UserId", how="left").merge(comp_feat, on="CompetitionId", how="left").fillna(0)

y_train = (train_ds["PrivateLeaderboardRank"] / train_ds["TotalTeams"]) <= 0.03

X_train = train.drop(columns=["UserId", "CompetitionId"])
X_test = test.drop(columns=["UserId", "CompetitionId"])
X_cols =[]
for col in X_train.columns:
    if col in X_test.columns:
        X_cols.append(col)
X_train = X_train[X_cols]
X_test = X_test[X_cols]

In [14]:
from catboost import CatBoostClassifier
cat = CatBoostClassifier(iterations=1000,learning_rate=0.03,depth=6,loss_function="Logloss", eval_metric="Logloss",random_seed=42,verbose=100,early_stopping_rounds=100, task_type = "GPU"
).fit(X_train, y_train)

0:	learn: 0.6379282	total: 82.4ms	remaining: 1m 22s
100:	learn: 0.1558452	total: 2.71s	remaining: 24.2s
200:	learn: 0.1499702	total: 5.36s	remaining: 21.3s
300:	learn: 0.1472267	total: 8.05s	remaining: 18.7s
400:	learn: 0.1452116	total: 10.7s	remaining: 16s
500:	learn: 0.1434504	total: 13.6s	remaining: 13.6s
600:	learn: 0.1420921	total: 16.4s	remaining: 10.9s
700:	learn: 0.1409218	total: 19s	remaining: 8.12s
800:	learn: 0.1398608	total: 21.7s	remaining: 5.39s
900:	learn: 0.1386397	total: 24.3s	remaining: 2.67s
999:	learn: 0.1377810	total: 26.9s	remaining: 0us


In [15]:
sample_sub= pd.read_csv("/kaggle/input/competitions/neoai-2026-day-1-kaggleforces/sample_submission.csv")
sample_sub["pred_score"]= cat.predict_proba(X_test)
sample_sub["pred_score"] =1-sample_sub["pred_score"]
sample_sub

,Id,pred_score
0,1659_184767,2.395009e-08
1,1659_475941,5.066865e-02
2,1659_147125,1.121789e-02
3,1659_13112,4.792632e-10
4,1659_114428,7.898798e-08
...,...,...
1895,676_340466,8.934267e-07
1896,676_41254,2.372497e-08
1897,676_469717,8.105950e-03
1898,676_301197,7.224471e-09


In [16]:
sample_sub.to_csv("kaggleforces.csv", index= False)